# Colab Runner: qwen_3_8b_single_agent
Automatically generated experiment runner. Configure your run parameters below.


## 0. Configuration Parameters
Set your experiment details here before running the rest of the notebook.


In [ ]:
# ----------------------------------------
# EDIT THESE VARIABLES FOR YOUR RUN
# ----------------------------------------

# The exact name of the dataset zip file mapped in your Google Drive (e.g. 'spider_dev', 'spider_train')
DATASET = "spider_dev"

# The name of the experiment (MUST match the exact folder name inside 'experiments/')
EXPERIMENT_NAME = "qwen_3_8b_single_agent"

# The name of the final prediction text file that will be saved and downloaded
PREDICTION_TXT_FILENAME = "qwen_3_8b_single_agent_colab_run"

# Execute a small sample instead of the full dataset (Requires --sample flag in main.py)
IS_SAMPLE = False

# Concurrency allowed for batch LLM requests
MAX_CONCURRENCY = 5

# Optional tagging for LangSmith / tracing purposes
EXPERIMENT_TAG = "qwen_3_8b_single_agent_v1"



## 1. Setup Environment & Clone Repository


In [ ]:
# Mount Google Drive to get access to the databases
from google.colab import drive
drive.mount('/content/drive')

# Clone the repository
!git clone https://github.com/ewerthonk/ai-agents-experiments.git ai_agents
%cd ai_agents

# Copy the dataset from Google Drive and strictly extract it into the spider data path
!mkdir -p data/spider
!cp /content/drive/MyDrive/{DATASET}.zip data/spider/
%cd data/spider/
!unzip -q {DATASET}.zip
!rm {DATASET}.zip
%cd /content/ai_agents


## 2. Install Dependencies using uv
We install `uv` explicitly to resolve `pyproject.toml` extremely fast.


In [ ]:
# Install uv
!pip install uv

# Use uv to install everything from pyproject.toml directly into the colab system env
!uv pip install --system -r pyproject.toml


## 3. Inject API Secrets


In [ ]:
import os
from google.colab import userdata

# Pull the secrets from Colab's native Secret Manager (the little key icon)
os.environ["LANGWATCH_API_KEY"] = userdata.get('LANGWATCH_API_KEY')
os.environ["LANGSMITH_TRACING"] = userdata.get('LANGSMITH_TRACING')
os.environ["LANGSMITH_ENDPOINT"] = userdata.get('LANGSMITH_ENDPOINT')
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["LANGSMITH_PROJECT"] = userdata.get('LANGSMITH_PROJECT')



## 4. Setup LLaMA Server
Compiles llama.cpp for CUDA support.


In [ ]:
# Ensure scripts are executable
!chmod +x experiments/{EXPERIMENT_NAME}/scripts/setup_llama_cuda.sh
!chmod +x experiments/{EXPERIMENT_NAME}/scripts/serve_llama_cuda.sh

# Run setup
!./experiments/{EXPERIMENT_NAME}/scripts/setup_llama_cuda.sh


## 5. Launch LLaMA Server (Background Subprocess)
Spawns the server globally so the notebook can continue executing.


In [ ]:
import subprocess
import time

print("Starting server globally in the background...")

# Launch globally so the next cell can read from 'process'
process = subprocess.Popen(
    [f"./experiments/{EXPERIMENT_NAME}/scripts/serve_llama_cuda.sh"], 
    stdout=subprocess.PIPE, 
    stderr=subprocess.STDOUT, 
    text=True
)

time.sleep(2)
print("Process spawned! Run the next cell to watch it load.")


Wait for the server to load by streaming the logs. This block will **automatically stop** when the server is ready.


In [ ]:
import time

print("Waiting for server to load the model into VRAM...")
print("-" * 50)

while True:
    line = process.stdout.readline()
    if not line:
        time.sleep(0.1)
        continue
    
    print(line, end="")
    
    # Automatically stop the cell log stream once the server finishes booting!
    if "update_slots: all slots are idle" in line:
        print("\n" + "-" * 50)
        print("✅ SUCCESS! Server is fully loaded and ready for LangChain!")
        break


## 6. Run the Experiment
Executes the LangGraph batch run using the parameters defined in Cell 0.


In [ ]:
# Ensure sampling string is formatted correctly for bash parsing
SAMPLE_FLAG = "--sample" if IS_SAMPLE else ""

# Run the main pipeline (using -u to force unbuffered logs to the console)
!python -u -m experiments.{EXPERIMENT_NAME}.main \
    --dataset "{DATASET}" \
    --prediction_txt_filename "{PREDICTION_TXT_FILENAME}" \
    --max_concurrency {MAX_CONCURRENCY} \
    --tag "{EXPERIMENT_TAG}" \
    {SAMPLE_FLAG}


## 7. Save & Download Results


In [ ]:
import sys
import importlib
from pathlib import Path
from google.colab import files

# Import your own main.py logic dynamically based on whatever EXPERIMENT_NAME is set to in Cell 0!
main_module = importlib.import_module(f"experiments.{EXPERIMENT_NAME}.main")
get_prediction_txt_path = main_module.get_prediction_txt_path

# Determine if main.py used the _sample dataset suffix
actual_dataset = f"{DATASET}_sample" if IS_SAMPLE else DATASET

# Replicate main.py's path resolution exactly
output_file = get_prediction_txt_path(actual_dataset, PREDICTION_TXT_FILENAME)

if output_file.exists():
    print(f"Found prediction file at: {{output_file}}")
    
    # Copy safely to drive
    !cp {{output_file}} /content/drive/MyDrive/
    
    # Trigger browser download
    files.download(str(output_file))
else:
    print(f"Error: Prediction file not found at {{output_file}}! Check the logs above.")
